In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Wczytanie danych ---
# Proba wczytania z Google Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    json_path = '/content/drive/Shareddrives/Sieci_Neuronowe/Projekt_Transformer_Grupa_3/wyniki_bonus_vit_medium_20ep.json'
    json_path2 = '/content/drive/Shareddrives/Sieci_Neuronowe/Projekt_Transformer_Grupa_3/wyniki_bonus2_vit_best_20ep.json'
    # Fallback - szukaj w roznych lokalizacjach
    import os
    if not os.path.exists(json_path) or not os.path.exists(json_path2):
        # Szukaj w MyDrive
        for root, dirs, files in os.walk('/content/drive'):
            if 'wyniki_bonus_vit_medium_20ep.json' in files:
                json_path = os.path.join(root, 'wyniki_bonus_vit_medium_20ep.json')
            if 'wyniki_bonus2_vit_best_20ep.json' in files:
                json_path2 = os.path.join(root, 'wyniki_bonus2_vit_best_20ep.json')
except:
    json_path = 'wyniki_bonus_vit_medium_20ep.json'
    json_path2 = 'wyniki_bonus2_vit_best_20ep.json'

with open(json_path, 'r', encoding='utf-8') as f:
    medium = json.load(f)
with open(json_path2, 'r', encoding='utf-8') as f:
    best = json.load(f)

print(f'Wczytano: {medium["experiment"]}')
print(f'Wczytano: {best["experiment"]}')


In [ ]:
import pandas as pd

# Configuration
config = pd.DataFrame({
    'Parameter': ['patch_size', 'embed_dim', 'num_heads', 'Nx', 'mlp_dim', 'epochs', 'lr'],
    'ViT_Medium': [medium['config']['patch_size'], medium['config']['embed_dim'], medium['config']['num_heads'],
                   medium['config']['Nx'], medium['config']['mlp_dim'], medium['config']['epochs'], medium['config']['lr']],
    'ViT_Best': [best['config']['patch_size'], best['config']['embed_dim'], best['config']['num_heads'],
                 best['config']['Nx'], best['config']['mlp_dim'], best['config']['epochs'], best['config']['lr']]
})
print('
Konfiguracja:')
print(config.to_string(index=False))


In [ ]:
# Results
results = pd.DataFrame({
    'Metric': ['Test Acc', 'Best Test Acc', 'Best Epoch', 'Train Acc', 'Params'],
    'ViT_Medium': [f"{medium['test_acc']:.2f}%", f"{medium['best_test_acc']:.2f}%", 
                   medium['best_epoch'], f"{medium['train_acc']:.2f}%", f"{medium['num_params']:,}"],
    'ViT_Best': [f"{best['test_acc']:.2f}%", f"{best['best_test_acc']:.2f}%", 
                 best['best_epoch'], f"{best['train_acc']:.2f}%", f"{best['num_params']:,}"]
})
print('
Wyniki:')
print(results.to_string(index=False))


In [ ]:
# Training curves
df_m = pd.DataFrame(medium['history'])
df_b = pd.DataFrame(best['history'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df_m['epoch'], df_m['loss'], 'o-', label='Medium', linewidth=2)
axes[0].plot(df_b['epoch'], df_b['loss'], 's-', label='Best', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_m['epoch'], df_m['train_acc'], 'o-', label='Medium', linewidth=2)
axes[1].plot(df_b['epoch'], df_b['train_acc'], 's-', label='Best', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Train Accuracy (%)')
axes[1].set_title('Training Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(df_m['epoch'], df_m['test_acc'], 'o-', label='Medium', linewidth=2)
axes[2].plot(df_b['epoch'], df_b['test_acc'], 's-', label='Best', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Test Accuracy (%)')
axes[2].set_title('Test Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Summary
print('
' + '='*70)
print('SUMMARY: ViT_Medium vs ViT_Best')
print('='*70)
print(f'Winner: ViT_Best ({best["best_test_acc"]:.2f}%)')
print(f'Improvement: +{best["best_test_acc"] - medium["best_test_acc"]:.2f}%')
print(f'
Key difference: patch_size {medium["config"]["patch_size"]} -> {best["config"]["patch_size"]}')
print(f'Params: {medium["num_params"]:,} -> {best["num_params"]:,}')
print('='*70)
